In [1]:
from helpers import init, preprocess_data

In [2]:
SEED = 42
X_train_raw, y_train_cm, X_val_raw, y_val_cm = init(SEED)
X_train, y_train, X_val, y_val, target_min_cm, target_range_cm = preprocess_data(X_train_raw, y_train_cm, X_val_raw, y_val_cm)

PyTorch: 2.11.0+cu130
Training rows: 42166
Validation rows: 4685
Input shape: (42166, 9)
Target x range (cm): 0.0 to 281.0
Target y range (cm): 0.0 to 275.0
Using split files:
  train: train_clean_3x3_1cm.csv 42166 rows
  validation: validation_clean_3x3_1cm.csv 4685 rows
RSS scale: 0.8493868112564087
Target min cm: [0. 0.]
Target range cm: [281. 275.]


# Get the model

In [3]:
from vlp_hackathon.baseline_model import BaselineMLP
from vlp_hackathon.improved_model import ImprovedMLP
model_cls = ImprovedMLP

# Run a trail

In [4]:
from torch import nn
import torch
import numpy as np
from matplotlib import pyplot as plt
import copy

def train_model(model, optimizer, loss_fn, epochs, batch_size, plot: bool = False, ret_model: bool = False):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    X_train_t = torch.from_numpy(X_train).to(device)
    y_train_t = torch.from_numpy(y_train).to(device)
    X_val_t = torch.from_numpy(X_val).to(device)
    y_val_t = torch.from_numpy(y_val).to(device)
    model = model.to(device)

    history = []
    best_model = copy.deepcopy(model)
    lowest_loss = float("inf")
    train_rng = np.random.default_rng(SEED)
    for epoch in range(epochs):
        model.train()
        permutation = train_rng.permutation(len(X_train))
        running = 0.0
        for start in range(0, len(permutation), batch_size):
            idx = permutation[start:start + batch_size]
            xb = X_train_t[idx]
            yb = y_train_t[idx]
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            if loss < lowest_loss:
                best_model = copy.deepcopy(model)
            optimizer.step()
            running += float(loss.item()) * len(idx)

        model.eval()
        with torch.no_grad():
            val_loss = float(loss_fn(model(X_val_t), y_val_t).item())
        history.append((running / len(X_train), val_loss))
        print(f"epoch={epoch + 1:02d} train_mse={history[-1][0]:.6f} val_mse={val_loss:.6f}")

    history = np.asarray(history, dtype=np.float32)
    if plot:
        plt.figure(figsize=(6, 4))
        plt.plot(history[:, 0], label="train")
        plt.plot(history[:, 1], label="validation")
        plt.xlabel("Epoch")
        plt.ylabel("MSE on normalized coordinates")
        plt.legend()
        plt.title("Baseline training curve")
        plt.show()

    if ret_model:
        return best_model
    return np.min(history[:, 1])

def run_trial(trial):
    epochs = 50

    print(trial)
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    batch_size = trial.suggest_int("batch_size", 16, 4096)

    depth = trial.suggest_int("depth", 1, 6)
    layers = []
    for layer in range(depth):
        width = trial.suggest_int(f"layer_{layer}", 2, 128)
        layers.append(width)

    # Start and end are fixed
    layers = [36] + layers + [2]

    act = trial.suggest_categorical("act", ["relu", "tanh", "sigmoid", ""])
    last_act = trial.suggest_categorical("last_act", ["relu", "tanh", "sigmoid", ""])
    layer_norm = trial.suggest_categorical("layer_norm", [True, False])

    model = model_cls(layers=layers, activation=act, last_act=last_act, layer_norm=layer_norm)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    return train_model(model, optimizer, loss_fn, epochs, batch_size)

In [ ]:
import optuna
from optuna.trial import TrialState

study = optuna.create_study(direction='minimize', study_name="task_1", load_if_exists=True, storage="sqlite:///trail_1.db")
study.optimize(run_trial, n_trials=5000, show_progress_bar=True)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

print("Study statistics: ")
print("  Number of finished trials: ", len(study.trials))
print("  Number of pruned trials: ", len(pruned_trials))
print("  Number of complete trials: ", len(complete_trials))

print("Best trial:")
trial = study.best_trial

print("  Value: ", trial.value)

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

[I 2026-07-08 16:49:20,846] Using an existing study with name 'task_1' instead of creating a new one.


  0%|          | 0/5000 [00:00<?, ?it/s]

Sequential(
  (0): Linear(in_features=9, out_features=109, bias=True)
  (1): Identity()
  (2): Linear(in_features=109, out_features=77, bias=True)
  (3): Identity()
  (4): Linear(in_features=77, out_features=62, bias=True)
  (5): Identity()
  (6): Linear(in_features=62, out_features=2, bias=True)
  (7): Identity()
)
epoch=01 train_mse=0.009066 val_mse=0.007282
epoch=02 train_mse=0.007926 val_mse=0.007538
epoch=03 train_mse=0.007795 val_mse=0.007643
epoch=04 train_mse=0.007764 val_mse=0.007141
epoch=05 train_mse=0.007652 val_mse=0.007169
epoch=06 train_mse=0.007627 val_mse=0.007105
epoch=07 train_mse=0.007585 val_mse=0.007038
epoch=08 train_mse=0.007573 val_mse=0.007236
epoch=09 train_mse=0.007577 val_mse=0.007331
epoch=10 train_mse=0.007552 val_mse=0.006962
epoch=11 train_mse=0.007548 val_mse=0.006986
epoch=12 train_mse=0.007529 val_mse=0.007134
epoch=13 train_mse=0.007514 val_mse=0.007161
epoch=14 train_mse=0.007536 val_mse=0.007114
epoch=15 train_mse=0.007523 val_mse=0.006960
epoch=1

In [ ]:
from plotly.io import show

fig = optuna.visualization.plot_optimization_history(study)
show(fig)

fig = optuna.visualization.plot_param_importances(study)
show(fig)

fig = optuna.visualization.plot_slice(study)
show(fig)

# Make the best model

In [ ]:
best_params = study.best_params
print(best_params)

layers = []
for layer_idx in range(best_params["depth"]):
    layer_name = f"layer_{layer_idx}"
    layer_size = best_params[layer_name]
    layers.append(layer_size)
layers = [36] + layers + [2]
best_model = model_cls(layers=layers, activation=best_params["act"], last_act=best_params["last_act"])

epochs = 40
batch_size = best_params["batch_size"]
optimizer = torch.optim.Adam(best_model.parameters(), lr=best_params["lr"])
loss_fn = nn.MSELoss()
best_model = train_model(best_model, optimizer, loss_fn, epochs, batch_size, plot=True, ret_model=True)

# Make the baseline model

In [ ]:
original_model = BaselineMLP(input_features=36)
history = []
epochs = 25
batch_size = 512
optimizer = torch.optim.Adam(original_model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()
original_model = train_model(original_model, optimizer, loss_fn, epochs, batch_size, plot=True, ret_model=True)

# Compare performance

In [ ]:
from vlp_hackathon.metrics import euclidean_errors_cm

def eval_model(model, name: str):
    model.eval()
    with torch.no_grad():
        float_norm = model(torch.from_numpy(X_val)).numpy()
    float_pred_cm = target_min_cm + float_norm * target_range_cm
    float_errors = euclidean_errors_cm(float_pred_cm, y_val_cm)

    print(f"\n{name}:")
    print(f"Float validation mean error:   {float_errors.mean():.3f} cm")
    print(f"Float validation median error: {np.median(float_errors):.3f} cm")
    print(f"Float validation p95 error:    {np.percentile(float_errors, 95):.3f} cm")
    return float_pred_cm

errors = []
for model, name in [(original_model, "original"), (best_model, "best")]:
    errors.append(eval_model(model, name))

plt.figure(figsize=(5, 5))
n_show = min(1500, len(y_val_cm))
plt.scatter(y_val_cm[:n_show, 0], y_val_cm[:n_show, 1], s=4, alpha=0.25, label="ground truth")
plt.scatter(errors[0][:n_show, 0], errors[0][:n_show, 1], s=4, alpha=0.25, label="pred_original")
plt.scatter(errors[1][:n_show, 0], errors[1][:n_show, 1], s=4, alpha=0.25, label="pred_improved")
plt.xlabel("x (cm)")
plt.ylabel("y (cm)")
plt.legend(markerscale=3)
plt.title("Clean validation positions")
plt.show()